# 09. File I/O and Text Processing

## What you'll learn

- How to **read and write files** using `open()` and `with` statements
- How to use **`pathlib`** for modern, cross-platform path handling
- How to work with **CSV and structured text formats**
- How to implement **text chunking strategies** (fixed-size with overlap, sentence-based, paragraph-based)
- How to **process multiple files** using `Path.glob()`
- How to build a **document preprocessing pipeline** that agents can use for retrieval

## Prerequisites

- [01 Python Fundamentals](01_python_fundamentals.ipynb) — variables, types, control flow
- [02 Functions and Scope](02_functions_and_scope.ipynb) — defining and calling functions
- [03 Data Structures](03_data_structures.ipynb) — lists, dicts, list-of-dicts pattern
- [04 Strings and JSON](04_strings_and_json.ipynb) — string methods, f-strings, json module
- [05 Error Handling](05_error_handling.ipynb) — try/except, context managers
- [07 Classes and OOP](07_classes_and_oop.ipynb) — helpful but not required

## Why this matters for agents

Agents that work with knowledge — research agents, RAG pipelines, document Q&A systems — need
to **read files from disk, split them into manageable chunks, and keep track of where each chunk
came from**. When you build a retrieval-augmented generation (RAG) agent in Core notebooks 10-11,
you'll rely on exactly the skills in this notebook: reading documents, splitting text into
overlapping chunks, and packaging each chunk with metadata (source file, chunk index) so the
agent can cite its sources.

> **Further reading:**
> - [Python `pathlib` docs](https://docs.python.org/3/library/pathlib.html) — the modern way to handle file paths
> - [Anthropic — Contextual Retrieval](https://www.anthropic.com/news/contextual-retrieval) — why chunking strategy matters for agent knowledge

---

## 1. Reading and Writing Files with `open()` and `with`

The most fundamental file operation in Python is `open()`. The `with` statement (a context
manager) ensures the file is properly closed after you're done — even if an error occurs.

Key modes:
- `"r"` — read (default)
- `"w"` — write (creates or **overwrites**)
- `"a"` — append
- `"x"` — create (fails if file exists)

Let's start by creating some sample files using `%%writefile` — a Jupyter magic command that
writes the cell contents directly to a file.

In [ ]:
%%writefile agent_log.txt
[2025-01-15 09:00:01] Agent started. Task: "Summarize recent news about AI safety."
[2025-01-15 09:00:02] Tool call: web_search(query="AI safety news January 2025")
[2025-01-15 09:00:05] Tool result: 3 articles found.
[2025-01-15 09:00:06] Tool call: read_article(url="https://example.com/ai-safety-update")
[2025-01-15 09:00:10] Tool result: Article text retrieved (2,340 words).
[2025-01-15 09:00:12] Generating summary...
[2025-01-15 09:00:18] Task complete. Final answer delivered to user.

Now let's read the file we just created.

In [ ]:
# Reading the entire file as a single string
with open("agent_log.txt", "r") as f:
    content = f.read()

print(f"Type: {type(content)}")
print(f"Length: {len(content)} characters")
print()
print(content)

In [ ]:
# Reading line by line — useful for large files
with open("agent_log.txt", "r") as f:
    for line_number, line in enumerate(f, start=1):
        # .strip() removes trailing newline
        print(f"  Line {line_number}: {line.strip()}")

In [ ]:
# .readlines() gives you a list of lines (each includes '\n')
with open("agent_log.txt", "r") as f:
    lines = f.readlines()

# Filter for just the tool calls
tool_calls = [line.strip() for line in lines if "Tool call" in line]
print(f"Found {len(tool_calls)} tool calls:")
for tc in tool_calls:
    print(f"  {tc}")

In [ ]:
# Writing a file: save agent messages as a simple log
messages = [
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."},
    {"role": "user", "content": "What about Germany?"},
    {"role": "assistant", "content": "The capital of Germany is Berlin."},
]

with open("conversation.txt", "w") as f:
    for msg in messages:
        f.write(f"[{msg['role'].upper()}] {msg['content']}\n")

# Verify what we wrote
with open("conversation.txt", "r") as f:
    print(f.read())

---

## 2. Modern File Handling with `pathlib`

The `pathlib` module (standard library, Python 3.4+) provides an object-oriented interface for
filesystem paths. It's the **recommended way** to work with files in modern Python.

Key advantages:
- **`/` operator** for joining paths (instead of `os.path.join()`)
- Methods like `.read_text()`, `.write_text()` — no need for `open()`
- `.exists()`, `.is_file()`, `.is_dir()` for checking path state
- `.stem`, `.suffix`, `.parent` for easy path introspection
- Works the same on Windows, Mac, and Linux

In [ ]:
from pathlib import Path

# Create a Path object
log_path = Path("agent_log.txt")

print(f"Path:      {log_path}")
print(f"Exists:    {log_path.exists()}")
print(f"Is file:   {log_path.is_file()}")
print(f"Name:      {log_path.name}")       # "agent_log.txt"
print(f"Stem:      {log_path.stem}")       # "agent_log"  (without extension)
print(f"Suffix:    {log_path.suffix}")     # ".txt"
print(f"Parent:    {log_path.parent}")     # directory containing the file
print(f"Absolute:  {log_path.resolve()}")  # full absolute path

In [ ]:
# The / operator joins paths — much cleaner than os.path.join()
data_dir = Path("agent_data")
tools_file = data_dir / "tools" / "definitions.txt"

print(f"Joined path: {tools_file}")
print(f"Parent:      {tools_file.parent}")

In [ ]:
# .read_text() and .write_text() — the one-liner approach
# No need for open() or context managers!

content = Path("agent_log.txt").read_text()
print(f"Read {len(content)} characters with .read_text()")

# Write a quick tool definition
tool_def = """name: web_search
description: Search the web for information
parameters:
  query: string (required)
  max_results: int (default 5)"""

Path("tool_definition.txt").write_text(tool_def)
print(f"Wrote tool definition to tool_definition.txt")
print(Path("tool_definition.txt").read_text())

---

## 3. Working with CSV and Structured Text Formats

Agents often need to process structured data from files — evaluation results, tool inventories,
or knowledge base entries. Python's `csv` module handles the most common tabular format.

In [ ]:
%%writefile tool_registry.csv
name,description,category,requires_api_key
web_search,Search the web for current information,retrieval,true
calculator,Evaluate mathematical expressions,computation,false
read_file,Read contents of a local file,file_io,false
send_email,Send an email to a specified address,communication,true
code_executor,Execute Python code in a sandbox,computation,false
image_generator,Generate images from text descriptions,creation,true

In [ ]:
import csv

# csv.DictReader turns each row into a dictionary
with open("tool_registry.csv", "r") as f:
    reader = csv.DictReader(f)
    tools = list(reader)

print(f"Loaded {len(tools)} tools\n")

# Each row is an OrderedDict with column names as keys
for tool in tools:
    api_note = " (needs API key)" if tool["requires_api_key"] == "true" else ""
    print(f"  {tool['name']:20s} [{tool['category']}]{api_note}")

In [ ]:
# Writing CSV — useful for saving evaluation results
eval_results = [
    {"task": "capital_lookup", "correct": True, "latency_ms": 230},
    {"task": "math_problem", "correct": True, "latency_ms": 450},
    {"task": "code_generation", "correct": False, "latency_ms": 1200},
    {"task": "summarization", "correct": True, "latency_ms": 890},
]

with open("eval_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["task", "correct", "latency_ms"])
    writer.writeheader()
    writer.writerows(eval_results)

# Verify
print(Path("eval_results.csv").read_text())

---

## 4. Text Chunking Strategies

This is where file I/O meets agent intelligence. When an agent needs to work with long documents,
it can't just shove the entire text into an LLM prompt — it needs to **split the text into
chunks** that fit within the context window.

Chunking strategy directly affects retrieval quality. There are three common approaches:

1. **Fixed-size with overlap** — split into chunks of N characters, with M characters of overlap
2. **Sentence-based** — split on sentence boundaries
3. **Paragraph-based** — split on blank lines (double newlines)

Let's create a longer document to work with.

In [ ]:
%%writefile ai_agents_overview.txt
What Are AI Agents?

An AI agent is a system that uses a large language model (LLM) as its core reasoning engine to autonomously accomplish tasks. Unlike simple chatbots that respond to one prompt at a time, agents can plan multi-step workflows, use external tools, and adapt their approach based on intermediate results.

The Core Agent Loop

The fundamental pattern behind all agents is surprisingly simple. The agent receives a task, thinks about what to do, takes an action (often calling a tool), observes the result, and then decides whether to continue or stop. This think-act-observe loop repeats until the task is complete. The key insight is that the LLM is making the decisions at each step, not following a hardcoded script.

Tools and Function Calling

Tools are what give agents their power. A tool is simply a function that the agent can choose to call. Common tools include web search, code execution, file reading, database queries, and API calls. The agent decides which tool to use based on the current state of the task. Modern LLMs are specifically trained to output structured tool calls that can be parsed and executed by the surrounding system.

Memory and Context

Agents need memory to work effectively. Short-term memory is the conversation history within a single session. Long-term memory involves storing and retrieving information across sessions, often using vector databases and embeddings. The challenge is that LLMs have finite context windows, so agents must be strategic about what information they keep in their working memory.

Retrieval-Augmented Generation

RAG is a technique where the agent retrieves relevant documents from a knowledge base before generating a response. The process involves three steps: first, the query is converted into an embedding vector. Second, similar documents are found using vector similarity search. Third, the retrieved documents are included in the prompt as context for the LLM to use when generating its answer. This allows agents to work with knowledge bases far larger than any context window.

Planning and Reasoning

Advanced agents can decompose complex tasks into subtasks, create execution plans, and revise those plans as new information emerges. The ReAct pattern combines reasoning traces with action execution, allowing the agent to think through problems step by step. Tree of Thought extends this by exploring multiple reasoning paths in parallel and selecting the most promising one.

Multi-Agent Systems

Some problems are best solved by multiple specialized agents working together. An orchestrator agent can delegate subtasks to specialist agents, each with their own tools and expertise. This mirrors how human teams work: a manager coordinates while specialists execute. The key challenge is communication between agents and aggregating their results into a coherent final output.

In [ ]:
document = Path("ai_agents_overview.txt").read_text()
print(f"Document length: {len(document)} characters")
print(f"Document lines:  {len(document.splitlines())}")

### Strategy 1: Fixed-size chunking with overlap

The simplest approach. Split the text every N characters, but overlap by M characters so that
sentences at chunk boundaries aren't cut off from their context.

In [ ]:
def chunk_fixed_size(text, chunk_size=300, overlap=50):
    """Split text into fixed-size chunks with overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap  # step forward, but keep overlap
    return chunks


chunks = chunk_fixed_size(document, chunk_size=300, overlap=50)
print(f"Fixed-size chunking: {len(chunks)} chunks\n")

for i, chunk in enumerate(chunks[:3]):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk[:100] + "...")
    print()

Notice how fixed-size chunks can split mid-sentence. This is the tradeoff: simple and
predictable, but chunks may not align with semantic boundaries.

### Strategy 2: Sentence-based chunking

Split on sentence boundaries, then group sentences into chunks of approximately N characters.
This preserves complete thoughts within each chunk.

In [ ]:
import re


def chunk_by_sentences(text, max_chunk_size=400):
    """Split text into chunks at sentence boundaries."""
    # Split on sentence-ending punctuation followed by whitespace
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        # If adding this sentence would exceed the limit, start a new chunk
        if current_chunk and len(current_chunk) + len(sentence) + 1 > max_chunk_size:
            chunks.append(current_chunk.strip())
            current_chunk = sentence
        else:
            current_chunk += (" " if current_chunk else "") + sentence

    # Don't forget the last chunk
    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks


chunks = chunk_by_sentences(document, max_chunk_size=400)
print(f"Sentence-based chunking: {len(chunks)} chunks\n")

for i, chunk in enumerate(chunks[:3]):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk[:120] + "...")
    print()

### Strategy 3: Paragraph-based chunking

Split on double newlines (paragraph breaks). This is the most semantically meaningful approach
when documents have clear paragraph structure — which is common for articles, reports, and docs.

In [ ]:
def chunk_by_paragraphs(text):
    """Split text on paragraph breaks (double newlines)."""
    paragraphs = re.split(r'\n\n+', text.strip())
    # Filter out empty strings and whitespace-only entries
    return [p.strip() for p in paragraphs if p.strip()]


chunks = chunk_by_paragraphs(document)
print(f"Paragraph-based chunking: {len(chunks)} chunks\n")

for i, chunk in enumerate(chunks):
    preview = chunk[:80].replace("\n", " ")
    print(f"  Chunk {i:2d} ({len(chunk):4d} chars): {preview}...")

Each paragraph is a clean, self-contained topic. But notice that some chunks are very short
(section headers) while others are long. For RAG, you often want to **merge the header with
its body paragraph** — we'll handle this in the capstone section.

---

## 5. Processing Multiple Files with `Path.glob()`

Agents working with knowledge bases need to process many files at once. `Path.glob()` lets
you find files matching a pattern — similar to shell wildcards.

Let's create a small collection of document files to work with.

In [ ]:
# Create a directory of knowledge base documents
kb_dir = Path("knowledge_base")
kb_dir.mkdir(exist_ok=True)

documents = {
    "tool_use.txt": (
        "Tool Use in AI Agents\n\n"
        "Tools are functions that an agent can invoke to interact with the outside world. "
        "Common tools include web search, code execution, file operations, and API calls.\n\n"
        "The agent selects which tool to use based on the current task and its reasoning. "
        "Modern LLMs are trained to output structured tool calls in JSON format, which the "
        "orchestration layer parses and executes."
    ),
    "memory_systems.txt": (
        "Memory Systems for Agents\n\n"
        "Short-term memory is the conversation history within a single session. It gives the "
        "agent context about what has been discussed and what actions have been taken.\n\n"
        "Long-term memory stores information across sessions using vector databases. The agent "
        "can retrieve relevant past experiences to inform current decisions. This is essential "
        "for agents that learn and improve over time."
    ),
    "planning.txt": (
        "Planning Strategies for Agents\n\n"
        "Plan-then-execute agents first create a complete plan, then carry out each step. This "
        "works well for tasks with clear structure.\n\n"
        "Adaptive planning agents revise their plan as new information emerges. They can "
        "handle unexpected results and change course when needed. The ReAct pattern is a "
        "popular approach that interleaves reasoning and action."
    ),
}

for filename, content in documents.items():
    (kb_dir / filename).write_text(content)

print(f"Created {len(documents)} files in {kb_dir}/")

In [ ]:
# Path.glob() finds files matching a pattern
txt_files = sorted(kb_dir.glob("*.txt"))
print(f"Found {len(txt_files)} .txt files:\n")

for path in txt_files:
    content = path.read_text()
    word_count = len(content.split())
    print(f"  {path.name:25s}  {word_count:4d} words  {len(content):4d} chars")

In [ ]:
# Practical pattern: load all documents into a list of dicts
# This is the format agents typically work with
def load_documents(directory, pattern="*.txt"):
    """Load all matching files from a directory into a list of dicts."""
    docs = []
    for path in sorted(Path(directory).glob(pattern)):
        docs.append({
            "source": path.name,
            "path": str(path),
            "text": path.read_text(),
        })
    return docs


docs = load_documents("knowledge_base")
print(f"Loaded {len(docs)} documents:\n")
for doc in docs:
    print(f"  {doc['source']:25s} ({len(doc['text'])} chars)")

---

## 6. Putting It Together: Document Preprocessing Pipeline

Now let's combine everything into a pipeline that an agent (or RAG system) would actually use.

The pipeline will:
1. Read all `.txt` files from a directory
2. Chunk each document using fixed-size chunks with overlap
3. Return a list of `{source, chunk_index, text}` dicts — ready for embedding and retrieval

In [ ]:
%%writefile knowledge_base/embeddings.txt
Understanding Embeddings

An embedding is a dense vector representation of text. Each piece of text — a word, sentence, or paragraph — is mapped to a list of floating-point numbers. Texts with similar meanings end up with similar vectors.

Embeddings are created by specialized neural networks trained on large text corpora. Popular embedding models include OpenAI's text-embedding-ada-002 and open-source alternatives like sentence-transformers.

In RAG systems, both the query and the knowledge base documents are converted to embeddings. The system then finds documents whose embedding vectors are closest to the query vector using cosine similarity or dot product. This is much more effective than keyword matching because it captures semantic meaning.

In [ ]:
def preprocess_documents(directory, chunk_size=300, overlap=50, pattern="*.txt"):
    """Full document preprocessing pipeline.

    Reads all matching files, chunks them with overlap, and returns
    a flat list of chunk dicts ready for embedding/retrieval.

    Args:
        directory: Path to the documents directory.
        chunk_size: Maximum characters per chunk.
        overlap: Characters of overlap between consecutive chunks.
        pattern: Glob pattern for matching files.

    Returns:
        List of dicts with keys: source, chunk_index, text
    """
    all_chunks = []

    for path in sorted(Path(directory).glob(pattern)):
        text = path.read_text().strip()
        if not text:
            continue

        # Chunk the document
        chunks = chunk_fixed_size(text, chunk_size=chunk_size, overlap=overlap)

        for idx, chunk_text in enumerate(chunks):
            all_chunks.append({
                "source": path.name,
                "chunk_index": idx,
                "text": chunk_text.strip(),
            })

    return all_chunks

In [ ]:
# Run the pipeline
chunks = preprocess_documents("knowledge_base", chunk_size=250, overlap=40)

print(f"Pipeline produced {len(chunks)} chunks from knowledge_base/\n")
print(f"{'Source':<25s} {'Chunk':>5s}  {'Chars':>5s}  Preview")
print("-" * 80)

for c in chunks:
    preview = c["text"][:45].replace("\n", " ")
    print(f"{c['source']:<25s} {c['chunk_index']:>5d}  {len(c['text']):>5d}  {preview}...")

In [ ]:
# This is exactly the data structure a RAG agent would work with.
# Let's look at one chunk in detail:
import json

sample = chunks[0]
print("Sample chunk (as the agent sees it):")
print(json.dumps(sample, indent=2))

This output format is exactly what you'll feed into an embedding model in
[10 NumPy for Embeddings](10_numpy_for_embeddings.ipynb) and then into a full RAG pipeline
in Core notebook 10.

---

## Try It Yourself

### Exercise 1: Word/Line/Character Counter

Write a function `file_stats(filepath)` that returns a dict with keys `"chars"`, `"words"`,
and `"lines"` for the given file. Test it on `ai_agents_overview.txt`.

```python
def file_stats(filepath):
    """Return {chars, words, lines} for the given file."""
    # Your code here
    pass

stats = file_stats("ai_agents_overview.txt")
print(stats)
# Expected output: something like {'chars': 2041, 'words': 315, 'lines': 21}
```

In [ ]:
# Exercise 1: Your code here


### Exercise 2: `chunk_by_sentences` with Overlap

Modify the `chunk_by_sentences` function to support an `overlap_sentences` parameter — the
number of sentences from the end of the previous chunk to repeat at the start of the next chunk.

```python
def chunk_by_sentences_overlap(text, max_chunk_size=400, overlap_sentences=1):
    """Chunk by sentences with N overlapping sentences between chunks."""
    # Your code here
    pass
```

Test it on `ai_agents_overview.txt` with `overlap_sentences=2`. Verify that the last 2 sentences
of chunk 0 appear at the start of chunk 1.

In [ ]:
# Exercise 2: Your code here


### Exercise 3: `DocumentLoader` Class with Keyword Search

Build a `DocumentLoader` class that:
1. Takes a directory path in `__init__` and loads all `.txt` files
2. Has a `.search(keyword)` method that returns a list of `{source, line_number, line}` dicts
   for every line containing the keyword (case-insensitive)
3. Has a `.summary()` method that prints the number of files loaded and total character count

```python
class DocumentLoader:
    def __init__(self, directory):
        # Your code here
        pass

    def search(self, keyword):
        # Your code here
        pass

    def summary(self):
        # Your code here
        pass

loader = DocumentLoader("knowledge_base")
loader.summary()
results = loader.search("agent")
for r in results[:5]:
    print(f"  {r['source']}:{r['line_number']} — {r['line'][:60]}...")
```

In [ ]:
# Exercise 3: Your code here


### Exercise 4: Paragraph-Aware Pipeline

Modify the `preprocess_documents` pipeline to use paragraph-based chunking instead of
fixed-size. Merge single-line paragraphs (like section headers) with the paragraph that
follows them, so each chunk contains both the heading and its content.

**Hint:** After splitting by `\n\n`, check if a paragraph has no period (`.`) — it's
probably a heading. Concatenate it with the next paragraph.

In [ ]:
# Exercise 4: Your code here


---

## Cleanup

Remove the temporary files created during this notebook.

In [ ]:
import shutil

# Remove individual files created by %%writefile and code cells
files_to_remove = [
    "agent_log.txt",
    "conversation.txt",
    "tool_definition.txt",
    "tool_registry.csv",
    "eval_results.csv",
    "ai_agents_overview.txt",
]

for filename in files_to_remove:
    path = Path(filename)
    if path.exists():
        path.unlink()
        print(f"  Removed {filename}")

# Remove the knowledge_base directory and all its contents
kb_path = Path("knowledge_base")
if kb_path.exists():
    shutil.rmtree(kb_path)
    print(f"  Removed {kb_path}/ directory")

print("\nCleanup complete.")